# House Price Prediction using Machine Learning
**Author:** Gaurav Singh Juneja  
**Project:** IBM Internship — House Price Prediction  
**Dataset:** California Housing Dataset (scikit-learn built-in)  
**Tech Stack:** Python, Scikit-learn, Pandas, NumPy, Matplotlib, Seaborn, Flask, Streamlit

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import os

print('All libraries imported successfully!')

## 2. Load and Explore the Dataset

In [ ]:
# Load California Housing Dataset
housing = fetch_california_housing(as_frame=True)
df = housing.frame

print('Dataset Shape:', df.shape)
print('\nFeature Names:', housing.feature_names)
print('\nTarget: MedHouseVal (Median House Value in $100,000s)')
df.head()

In [ ]:
# Dataset Info
print('Dataset Info:')
df.info()

In [ ]:
# Statistical Summary
print('Statistical Summary:')
df.describe()

In [ ]:
# Check for missing values
print('Missing Values:')
print(df.isnull().sum())
print('\nTotal missing values:', df.isnull().sum().sum())

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Distribution of target variable
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['MedHouseVal'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Median House Value', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Median House Value ($100,000s)')
axes[0].set_ylabel('Frequency')

axes[1].boxplot(df['MedHouseVal'], patch_artist=True,
                boxprops=dict(facecolor='steelblue', color='navy'))
axes[1].set_title('Box Plot of Median House Value', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Median House Value ($100,000s)')

plt.tight_layout()
plt.savefig('target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Target variable distribution plotted.')

In [ ]:
# Feature distributions
features = housing.feature_names
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(features):
    axes[i].hist(df[col], bins=40, color='coral', edgecolor='white')
    axes[i].set_title(f'Distribution of {col}', fontsize=10, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Feature distributions plotted.')

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(12, 9))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap of All Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Correlation heatmap plotted.')

In [ ]:
# Scatter plots: Top correlated features vs target
top_features = corr['MedHouseVal'].drop('MedHouseVal').abs().nlargest(4).index.tolist()
print('Top correlated features:', top_features)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for i, feat in enumerate(top_features):
    axes[i].scatter(df[feat], df['MedHouseVal'], alpha=0.3, s=5, color='teal')
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('MedHouseVal')
    axes[i].set_title(f'{feat} vs MedHouseVal', fontweight='bold')

plt.tight_layout()
plt.savefig('scatter_plots.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Data Preprocessing

In [ ]:
# Feature Engineering: Add derived features
df['RoomsPerHousehold'] = df['AveRooms'] / df['HouseAge'].replace(0, 1)
df['BedroomsPerRoom'] = df['AveBedrms'] / df['AveRooms'].replace(0, 1)
df['PopulationPerHousehold'] = df['Population'] / df['AveOccup'].replace(0, 1)

print('Feature engineering complete. New shape:', df.shape)
df[['RoomsPerHousehold', 'BedroomsPerRoom', 'PopulationPerHousehold']].describe()

In [ ]:
# Prepare features and target
X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

print('Features shape:', X.shape)
print('Target shape:', y.shape)
print('\nFeature columns:', list(X.columns))

In [ ]:
# Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set size: {X_train.shape[0]} samples')
print(f'Test set size:     {X_test.shape[0]} samples')

In [ ]:
# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('Feature scaling complete.')
print('Train mean (first 3 cols):', X_train_scaled[:, :3].mean(axis=0).round(4))
print('Train std  (first 3 cols):', X_train_scaled[:, :3].std(axis=0).round(4))

## 5. Model Training and Evaluation

In [ ]:
def evaluate_model(model, X_tr, X_te, y_tr, y_te, model_name):
    """Train model and return evaluation metrics."""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    
    mae  = mean_absolute_error(y_te, y_pred)
    mse  = mean_squared_error(y_te, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_te, y_pred)
    
    print(f'\n=== {model_name} ===')
    print(f'  MAE  : {mae:.4f}')
    print(f'  RMSE : {rmse:.4f}')
    print(f'  R²   : {r2:.4f}')
    
    return {'Model': model_name, 'MAE': mae, 'RMSE': rmse, 'R2': r2, 'y_pred': y_pred}

results = {}

In [ ]:
# Linear Regression
lr = LinearRegression()
results['Linear Regression'] = evaluate_model(lr, X_train_scaled, X_test_scaled, y_train, y_test, 'Linear Regression')

In [ ]:
# Ridge Regression
ridge = Ridge(alpha=1.0)
results['Ridge Regression'] = evaluate_model(ridge, X_train_scaled, X_test_scaled, y_train, y_test, 'Ridge Regression')

In [ ]:
# Lasso Regression
lasso = Lasso(alpha=0.01)
results['Lasso Regression'] = evaluate_model(lasso, X_train_scaled, X_test_scaled, y_train, y_test, 'Lasso Regression')

In [ ]:
# Decision Tree Regressor
dt = DecisionTreeRegressor(max_depth=8, random_state=42)
results['Decision Tree'] = evaluate_model(dt, X_train_scaled, X_test_scaled, y_train, y_test, 'Decision Tree')

In [ ]:
# Random Forest Regressor
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
results['Random Forest'] = evaluate_model(rf, X_train_scaled, X_test_scaled, y_train, y_test, 'Random Forest')

In [ ]:
# Gradient Boosting Regressor
gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42)
results['Gradient Boosting'] = evaluate_model(gb, X_train_scaled, X_test_scaled, y_train, y_test, 'Gradient Boosting')

## 6. Model Comparison

In [ ]:
# Build comparison DataFrame
metrics_df = pd.DataFrame([
    {'Model': k, 'MAE': v['MAE'], 'RMSE': v['RMSE'], 'R2': v['R2']}
    for k, v in results.items()
]).sort_values('R2', ascending=False).reset_index(drop=True)

print('Model Comparison (sorted by R² score):')
metrics_df

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

for ax, metric in zip(axes, ['MAE', 'RMSE', 'R2']):
    vals = metrics_df[metric]
    bars = ax.bar(metrics_df['Model'], vals, color=colors[:len(metrics_df)])
    ax.set_title(f'{metric} Comparison', fontsize=12, fontweight='bold')
    ax.set_ylabel(metric)
    ax.set_xticklabels(metrics_df['Model'], rotation=30, ha='right')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Model comparison chart saved.')

## 7. Best Model — Hyperparameter Tuning (Random Forest)

In [ ]:
# Hyperparameter tuning for Random Forest (best model)
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid, cv=3, scoring='r2', n_jobs=-1, verbose=1
)
grid_search.fit(X_train_scaled, y_train)

print('Best parameters:', grid_search.best_params_)
print('Best CV R² score:', round(grid_search.best_score_, 4))

In [ ]:
# Evaluate best tuned model
best_model = grid_search.best_estimator_
tuned_results = evaluate_model(best_model, X_train_scaled, X_test_scaled, y_train, y_test, 'Tuned Random Forest')

y_pred_best = tuned_results['y_pred']
print('\nTuned model evaluation complete.')

## 8. Residual Analysis and Prediction Plots

In [ ]:
# Actual vs Predicted
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(y_test, y_pred_best, alpha=0.4, s=10, color='steelblue')
min_val, max_val = min(y_test.min(), y_pred_best.min()), max(y_test.max(), y_pred_best.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Values')
axes[0].set_ylabel('Predicted Values')
axes[0].set_title('Actual vs Predicted (Tuned RF)', fontsize=13, fontweight='bold')
axes[0].legend()

# Residuals
residuals = y_test.values - y_pred_best
axes[1].scatter(y_pred_best, residuals, alpha=0.4, s=10, color='coral')
axes[1].axhline(y=0, color='black', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Predicted Values')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residual Plot (Tuned RF)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Feature Importance

In [ ]:
# Feature importance from best model
importances = best_model.feature_importances_
feat_names = X.columns.tolist()
feat_imp_df = pd.DataFrame({'Feature': feat_names, 'Importance': importances})
feat_imp_df = feat_imp_df.sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 7))
plt.barh(feat_imp_df['Feature'], feat_imp_df['Importance'], color='teal')
plt.xlabel('Importance Score')
plt.title('Feature Importance — Tuned Random Forest', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 5 features:')
print(feat_imp_df.sort_values('Importance', ascending=False).head())

## 10. Save the Model and Scaler

In [ ]:
# Save model and scaler
os.makedirs('models', exist_ok=True)
joblib.dump(best_model, 'models/house_price_model.pkl')
joblib.dump(scaler, 'models/scaler.pkl')

print('Model saved to models/house_price_model.pkl')
print('Scaler saved to models/scaler.pkl')

## 11. Sample Prediction (Inference)

In [ ]:
# Load saved model and make a prediction
loaded_model = joblib.load('models/house_price_model.pkl')
loaded_scaler = joblib.load('models/scaler.pkl')

# Sample input (using first test sample)
sample = X_test.iloc[[0]]
sample_scaled = loaded_scaler.transform(sample)
prediction = loaded_model.predict(sample_scaled)[0]

print('Sample Input:')
print(sample.to_string())
print(f'\nActual Value   : ${y_test.iloc[0] * 100000:,.0f}')
print(f'Predicted Value: ${prediction * 100000:,.0f}')

## 12. Final Summary

In [ ]:
print('='*60)
print('       HOUSE PRICE PREDICTION — FINAL SUMMARY')
print('='*60)
print(f'Dataset        : California Housing (scikit-learn)')
print(f'Samples        : {df.shape[0]:,}')
print(f'Features Used  : {X.shape[1]}')
print(f'Best Model     : Tuned Random Forest Regressor')
print(f'  MAE  : {tuned_results["MAE"]:.4f}')
print(f'  RMSE : {tuned_results["RMSE"]:.4f}')
print(f'  R²   : {tuned_results["R2"]:.4f}')
print('='*60)

metrics_df